In [1]:
# ==========================================
# 1. TEMİZ VE GÜNCEL KURULUM
# ==========================================
import os

print("⚙️ Eski dosyalar temizleniyor ve Google Chrome indiriliyor...")

# Önceki kurulumları temizle
os.system("rm -rf /usr/bin/google-chrome")
os.system("rm -rf /usr/bin/chromedriver")

# Gerekli kütüphaneleri kur
os.system("pip install selenium supabase beautifulsoup4 webdriver-manager -qq")

# 1. Google Chrome'un Orijinal .deb dosyasını indir ve kur
# (Chromium değil, Google Chrome Stable kullanıyoruz)
os.system("wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb")
os.system("apt-get update -qq")
os.system("apt-get install -y ./google-chrome-stable_current_amd64.deb -qq")

print("✅ Kurulum tamamlandı.")

⚙️ Eski dosyalar temizleniyor ve Google Chrome indiriliyor...
✅ Kurulum tamamlandı.


In [2]:
# ==========================================
# 2. DRIVER AYARLARI (OTO-GÜNCELLEMELİ)
# ==========================================
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

# Ayarlar
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--window-size=1920,1080')
chrome_options.add_argument('--remote-debugging-port=9222') # Çökme önleyici port

# Driver'ı otomatik indir ve ayarla (En kritik kısım burası)
# ChromeDriverManager().install() komutu yüklü Chrome'a uygun driver'ı bulur.
service = Service(ChromeDriverManager().install())

# Test için tarayıcıyı başlatıp kapatıyoruz
try:
    driver_test = webdriver.Chrome(service=service, options=chrome_options)
    print("✅ Tarayıcı başarıyla başlatıldı! Sorun çözüldü.")
    driver_test.quit()
except Exception as e:
    print(f"❌ Hala hata var: {e}")

✅ Tarayıcı başarıyla başlatıldı! Sorun çözüldü.


In [ ]:
# ==========================================
# 3. SUPABASE BAĞLANTISI VE KEYLER
# ==========================================
import os
from supabase import create_client, Client

# Ortam degiskenlerinden oku
MY_SUPABASE_URL = os.getenv("SUPABASE_URL")
MY_SUPABASE_KEY = os.getenv("SUPABASE_KEY")

if not MY_SUPABASE_URL or not MY_SUPABASE_KEY:
    raise ValueError("SUPABASE_URL veya SUPABASE_KEY eksik. Ortam degiskenlerini ayarla.")

try:
    # İstemciyi oluşturuyoruz
    supabase: Client = create_client(MY_SUPABASE_URL, MY_SUPABASE_KEY)
    print("✅ Supabase bağlantısı hazır.")
except Exception as e:
    print(f"❌ Bağlantı Hatası: {e}")

✅ Supabase bağlantısı hazır.


In [ ]:
# ==========================================
# 3. SCRAPING MOTORU (V9 - ID PARAMETRESİ UYUMLU)
# ==========================================
import re
import ast
import time
from datetime import datetime
from bs4 import BeautifulSoup
from selenium import webdriver
from urllib.parse import urlparse, parse_qs
import copy

def clean_odd(value):
    if not value or value == '-' or value == '': return None
    try: return float(value.replace(',', '.'))
    except: return None

def parse_date(date_str):
    try:
        clean_str = date_str.replace('Tarih :', '').strip()
        dt_obj = datetime.strptime(clean_str, '%d.%m.%Y %H:%M')
        return dt_obj.strftime('%Y-%m-%d %H:%M:%S')
    except: return None

# Ana Fonksiyon
def process_full_match(match_url):
    chrome_options.page_load_strategy = 'eager'
    # service ve chrome_options Hücre 1'den geliyor olmalı
    driver = webdriver.Chrome(service=service, options=chrome_options)
    
    try:
        print(f"🌍 Veri çekiliyor: {match_url}")
        driver.get(match_url)
        time.sleep(2) 
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # --- 1. MAÇ KODU ---
        match_code = "0"
        try:
            parsed_url = urlparse(match_url)
            query_params = parse_qs(parsed_url.query)
            if 'id' in query_params:
                match_code = query_params['id'][0]
            else:
                match_code = match_url.lower().split('/mac/')[1].split('/')[0]
        except: 
            match_code = str(int(time.time()))

        # --- 2. MAÇ DETAYLARI ---
        match_info = {"league": None, "season": None, "match_date": None}
        info_wrapper = soup.find("div", class_="match-info-wrapper-top")
        
        if info_wrapper:
            season_div = info_wrapper.find("div", class_="match-info-wrapper-season")
            if season_div:
                full_text = season_div.get_text(strip=True)
                match = re.search(r'(\d{4}/\d{4})', full_text)
                if match:
                    match_info["season"] = match.group(1)
                    match_info["league"] = full_text.replace(match.group(1), "").strip()
                else:
                    match_info["league"] = full_text

            date_div = info_wrapper.find("div", class_="match-info-date")
            if date_div:
                match_info["match_date"] = parse_date(date_div.get_text(strip=True))

        # --- 3. SKOR VE DURUM ---
        match_status = "Bilinmiyor"
        score_ft = None
        score_ht = None 
        
        try:
            status_div = soup.find("div", id="dvStatusText") or soup.find("div", class_="match-time")
            if status_div: match_status = status_div.get_text(strip=True)
            
            ft_div = soup.find("div", id="dvScoreText") or soup.find("div", class_="match-score")
            if ft_div: score_ft = ft_div.get_text(strip=True)

            ht_div = soup.find("div", id="dvHTScoreText") or soup.find("div", class_="hf-match-score")
            if ht_div:
                raw_ht = ht_div.get_text(strip=True)
                # İY, :, (, ) temizliği
                score_ht = raw_ht.replace('İY', '').replace(':', '').replace('(', '').replace(')', '').strip()
        except: pass

        # --- 4. TAKIMLAR ---
        home, away = "Bilinmiyor", "Bilinmiyor"
        try:
            h_tag = soup.find("a", class_="left-block-team-name")
            a_tag = soup.find("a", class_="r-left-block-team-name")
            if h_tag: home = h_tag.get_text(strip=True)
            if a_tag: away = a_tag.get_text(strip=True)
        except: pass
        
        print(f"📌 {home} vs {away} | MS: {score_ft} / İY: {score_ht}")

        row = {
            "match_code": match_code,
            "home_team": home, "away_team": away,
            "league": match_info["league"], "season": match_info["season"],
            "match_date": match_info["match_date"],
            "score_ft": score_ft,
            "score_ht": score_ht,
            "status": match_status
        }

        # --- 5. ORANLAR ---
        bet_boxes = soup.find_all("div", class_="md")
        
        for box in bet_boxes:
            title_div = box.find("div", class_="detail-title")
            if not title_div: continue
            
            title_copy = copy.copy(title_div)
            if title_copy.find("span"):
                title_copy.find("span").decompose()
            
            market_name = title_copy.get_text(strip=True).replace(',', '.')
            link = box.find("a", href=True)
            
            if link and "openOddsDialog" in link['href']:
                match = re.search(r"openOddsDialog\((.*?)\)", link['href'])
                if match:
                    params = match.group(1)
                    arrays = re.findall(r"\[.*?\]", params)
                    if len(arrays) >= 2:
                        outcomes = ast.literal_eval(arrays[0])
                        raw_odds = ast.literal_eval(arrays[1])
                        odds_map = dict(zip(outcomes, raw_odds))

                        if market_name == "Maç Sonucu":
                            row["ms_1"] = clean_odd(odds_map.get('1'))
                            row["ms_x"] = clean_odd(odds_map.get('X'))
                            row["ms_2"] = clean_odd(odds_map.get('2'))
                        elif market_name == "Çifte Şans":
                            row["cs_1x"] = clean_odd(odds_map.get('1-X'))
                            row["cs_12"] = clean_odd(odds_map.get('1-2'))
                            row["cs_x2"] = clean_odd(odds_map.get('X-2'))
                        elif market_name == "İlk Yarı/Maç Sonucu":
                            row["iyms_11"] = clean_odd(odds_map.get('1/1'))
                            row["iyms_1x"] = clean_odd(odds_map.get('1/X'))
                            row["iyms_12"] = clean_odd(odds_map.get('1/2'))
                            row["iyms_x1"] = clean_odd(odds_map.get('X/1'))
                            row["iyms_xx"] = clean_odd(odds_map.get('X/X'))
                            row["iyms_x2"] = clean_odd(odds_map.get('X/2'))
                            row["iyms_21"] = clean_odd(odds_map.get('2/1'))
                            row["iyms_2x"] = clean_odd(odds_map.get('2/X'))
                            row["iyms_22"] = clean_odd(odds_map.get('2/2'))
                        elif "1.5 Alt/Üst" in market_name:
                            if "1. Yarı" not in market_name and "Evsahibi" not in market_name and "Deplasman" not in market_name:
                                row["au_15_alt"] = clean_odd(odds_map.get('Alt'))
                                row["au_15_ust"] = clean_odd(odds_map.get('Üst'))
                        elif "2.5 Alt/Üst" in market_name:
                            if "1. Yarı" not in market_name and "Evsahibi" not in market_name and "Deplasman" not in market_name:
                                row["au_25_alt"] = clean_odd(odds_map.get('Alt'))
                                row["au_25_ust"] = clean_odd(odds_map.get('Üst'))
                        elif market_name == "Karşılıklı Gol":
                            row["kg_var"] = clean_odd(odds_map.get('Var'))
                            row["kg_yok"] = clean_odd(odds_map.get('Yok'))
                        elif market_name == "Toplam Gol Aralığı":
                            row["tg_0_1"] = clean_odd(odds_map.get('0-1 Gol'))
                            row["tg_2_3"] = clean_odd(odds_map.get('2-3 Gol'))
                            row["tg_4_5"] = clean_odd(odds_map.get('4-5 Gol'))
                            row["tg_6_plus"] = clean_odd(odds_map.get('6+ Gol'))

        data = supabase.table("matches").upsert(row).execute()
        print(f"✅ KAYDEDİLDİ.")

    except Exception as e:
        print(f"❌ Hata: {e}")
    finally:
        driver.quit()

🌍 Kontrol ediliyor: https://arsiv.mackolik.com/Mac/4308325/Gaziantep-FK-Galatasaray
📌 Gaziantep FK vs Galatasaray | MS: 0 - 3 | İY: 0 - 3
✅ KAYDEDİLDİ: Gaziantep FK vs Galatasaray (MS: 0 - 3 / İY: 0 - 3)


In [6]:
# ==========================================
# 4. LİNK TOPLAYICI (API - REQUESTS ONLY & CLEANER)
# ==========================================
import requests
import ast
import time

def get_season_links_fast(season_id, total_weeks=38):
    print(f"🚀 Hızlı tarama başlıyor (Sezon: {season_id})...\n")
    
    all_match_links = []
    base_url = "https://arsiv.mackolik.com/AjaxHandlers/FixtureHandler.aspx"
    
    # Standart bir tarayıcı gibi görünmek için Header
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Referer": "https://arsiv.mackolik.com/"
    }

    for week in range(1, total_weeks + 1):
        params = {
            "command": "getMatches",
            "id": season_id,
            "week": week
        }
        
        try:
            response = requests.get(base_url, params=params, headers=headers, timeout=10)
            
            if response.status_code == 200 and response.text.strip():
                # --- TEMİZLİK İŞLEMİ (PYTHON UYUMLU HALE GETİRME) ---
                raw_data = response.text
                
                # 1. 'null' değerlerini 'None' yap
                raw_data = raw_data.replace('null', 'None')
                
                # 2. Boş alanları (,,) düzelt. Çift virgül kalmayana kadar döngüye sokuyoruz.
                # Örnek: [1,,2] -> [1,None,2]
                while ',,' in raw_data:
                    raw_data = raw_data.replace(',,', ',None,')
                
                # Listenin başındaki veya sonundaki virgül hatalarını düzeltmek için (Nadir durum)
                raw_data = raw_data.replace('[,', '[None,').replace(',]', ',None]')

                try:
                    # Artık Python listesine çevirebiliriz
                    matches_data = ast.literal_eval(raw_data)
                    
                    week_count = 0
                    for match in matches_data:
                        # match[0] -> Maç ID
                        # match[2] -> Durum (MS, Ert, vs.)
                        
                        if len(match) > 3: # Hata olmasın diye uzunluk kontrolü
                            match_id = match[0]
                            status = match[2]
                            
                            # SADECE BİTMİŞ MAÇLAR
                            if status == "MS":
                                # Yeni link formatı: Default.aspx?id=...
                                full_link = f"https://arsiv.mackolik.com/Match/Default.aspx?id={match_id}"
                                if full_link not in all_match_links:
                                    all_match_links.append(full_link)
                                    week_count += 1
                    
                    print(f"✅ {week}. Hafta: {week_count} maç eklendi.")
                    
                except Exception as parse_error:
                    print(f"⚠️ {week}. Hafta ayrıştırma hatası: {parse_error}")
                    # Hata ayıklama için bozuk veriyi görebilirsin:
                    # print(f"Hatalı Veri: {raw_data[:100]}...") 
            else:
                print(f"⚠️ {week}. Hafta boş döndü.")
                
        except Exception as e:
            print(f"❌ {week}. Hafta bağlantı hatası: {e}")
            
        time.sleep(0.2) # Sunucuyu yormamak için minik bekleme

    print(f"\n🎉 TOPLAM: {len(all_match_links)} adet bitmiş maç toplandı.")
    return all_match_links

# --- BURASI ÇOK ÖNEMLİ: FONKSİYONU ÇAĞIRMA KISMI ---
# Bu kısmı Hücre 4'ün en altına ekle ve Hücre 4'ü tekrar çalıştır.

SEZON_ID = 70381  # 2025/2026 Sezonu
TOPLAM_HAFTA = 38 

# Fonksiyonu çalıştırıp sonucu 'match_list' değişkenine atıyoruz
match_list = get_season_links_fast(SEZON_ID, TOPLAM_HAFTA)

# Kontrol edelim, içi dolmuş mu?
print(f"------------\nListe Durumu: {len(match_list)} maç hazır.")
if len(match_list) > 0:
    print("✅ Şimdi Hücre 5'i çalıştırabilirsin.")
else:
    print("❌ Liste boş geldi! Sezon ID'sini kontrol et.")

🚀 Hızlı tarama başlıyor (Sezon: 70381)...

✅ 1. Hafta: 9 maç eklendi.
✅ 2. Hafta: 9 maç eklendi.
✅ 3. Hafta: 9 maç eklendi.
✅ 4. Hafta: 9 maç eklendi.
✅ 5. Hafta: 9 maç eklendi.
✅ 6. Hafta: 9 maç eklendi.
✅ 7. Hafta: 9 maç eklendi.
✅ 8. Hafta: 9 maç eklendi.
✅ 9. Hafta: 9 maç eklendi.
✅ 10. Hafta: 9 maç eklendi.
✅ 11. Hafta: 9 maç eklendi.
✅ 12. Hafta: 9 maç eklendi.
✅ 13. Hafta: 9 maç eklendi.
✅ 14. Hafta: 9 maç eklendi.
✅ 15. Hafta: 9 maç eklendi.
✅ 16. Hafta: 9 maç eklendi.
✅ 17. Hafta: 9 maç eklendi.
✅ 18. Hafta: 0 maç eklendi.
✅ 19. Hafta: 0 maç eklendi.
✅ 20. Hafta: 0 maç eklendi.
✅ 21. Hafta: 0 maç eklendi.
✅ 22. Hafta: 0 maç eklendi.
✅ 23. Hafta: 0 maç eklendi.
✅ 24. Hafta: 0 maç eklendi.
✅ 25. Hafta: 0 maç eklendi.
✅ 26. Hafta: 0 maç eklendi.
✅ 27. Hafta: 0 maç eklendi.
✅ 28. Hafta: 0 maç eklendi.
✅ 29. Hafta: 0 maç eklendi.
✅ 30. Hafta: 0 maç eklendi.
✅ 31. Hafta: 0 maç eklendi.
✅ 32. Hafta: 0 maç eklendi.
✅ 33. Hafta: 0 maç eklendi.
✅ 34. Hafta: 0 maç eklendi.
✅ 35. Hafta: 0

In [7]:
# ==========================================
# 5. TOPLU KAYIT BAŞLATICI
# ==========================================

def start_batch_processing(links):
    total = len(links)
    print(f"🚀 {total} adet maç işlenmeye başlıyor...")
    
    for i, link in enumerate(links):
        print(f"\n[{i+1}/{total}] ------------------------------------------------")
        try:
            process_full_match(link) # Hücre 3'teki V9 fonksiyonu
        except Exception as e:
            print(f"⚠️ Genel döngü hatası: {e}")
        
        time.sleep(1) # Nezaketen bekleme

# --- BAŞLAT ---
if 'match_list' in globals() and len(match_list) > 0:
    start_batch_processing(match_list)
else:
    print("⚠️ Önce Hücre 4'ü çalıştırıp linkleri topla!")

🚀 153 adet maç işlenmeye başlıyor...

[1/153] ------------------------------------------------
⚠️ Genel döngü hatası: name 'process_full_match' is not defined

[2/153] ------------------------------------------------
⚠️ Genel döngü hatası: name 'process_full_match' is not defined

[3/153] ------------------------------------------------
⚠️ Genel döngü hatası: name 'process_full_match' is not defined

[4/153] ------------------------------------------------
⚠️ Genel döngü hatası: name 'process_full_match' is not defined

[5/153] ------------------------------------------------
⚠️ Genel döngü hatası: name 'process_full_match' is not defined

[6/153] ------------------------------------------------
⚠️ Genel döngü hatası: name 'process_full_match' is not defined


KeyboardInterrupt: 